# Measurement conditions during the Tier 1 grid

From 11 Nov 2026 Tailscale held about one core of this Mac (macOS reports in `/Library/Logs/DiagnosticReports/Tailscale_2026-11-1*`). The runner's hygiene check sees sleep, power and the lid, not other processes' load. Cell 1 compares the grid's medians with earlier runs of the same cells. Cell 2 checks S3 at 1 GiB offline, whose rerun after the lid closed was much slower than S3's scaling predicts.

In [1]:
# Grid medians against earlier runs of the same (arm, db, record, phase), warmups and faulted rows left out.
# Offline: server_preproc_s. Online: server_answer_s (the slower server for two-server arms).
# The feasibility run (11 Nov from 12:19 IST) took one repetition per cell, so only its offline phases
# and S4's online phase (5 repetitions in the 10 Nov S4 run) are compared.
import json, statistics as st
from collections import defaultdict
from pathlib import Path
import pandas as pd

raw = Path.cwd().parent / "results/raw"
MET = {"offline": "server_preproc_s", "online": "server_answer_s"}

def medians(files):
    acc = defaultdict(list)
    for f in files:
        for line in f.read_text().splitlines():
            r = json.loads(line)
            if r.get("is_warmup") or r.get("status") != "ok" or r.get("faulted") or r.get("fault", "none") != "none":
                continue
            v = r["metrics"].get(MET[r["phase"]])
            v = max(v) if isinstance(v, list) else v
            if v:
                acc[(r["arm"], r["db_bytes"], r["record_bytes"], r["phase"])].append(v)
    return {k: (st.median(v), len(v)) for k, v in acc.items()}

grid = medians(sorted((raw / "all-20261112_tier1").glob("*_*_*_*.jsonl")))
refs = {"feasibility 11 Nov": "grid-feasibility-20261111T064956Z", "S4 cost 10 Nov": "s4-cost-20261110T164450Z"}
rows = []
for name, d in refs.items():
    files = [f for f in sorted((raw / d).glob("*.jsonl")) if f.stem not in ("hygiene", "s4_cells", "s4_costs")]
    for k, (m, n) in medians(files).items():
        if k in grid and (k[3] == "offline" or n >= 5):
            rows.append({"reference": name, "arm": k[0], "db_MiB": k[1] / 2**20, "record_bytes": k[2], "phase": k[3],
                         "ref_s": m, "ref_n": n, "grid_s": grid[k][0], "grid_n": grid[k][1], "ratio": grid[k][0] / m})
df = pd.DataFrame(rows)
summary = df.groupby(["reference", "phase", "arm"]).ratio.agg(["count", "min", "median", "max"]).round(3)
print(summary.to_string())
print(f"\nall compared cells: ratio grid / earlier from {df.ratio.min():.3f} to {df.ratio.max():.3f}, median {df.ratio.median():.3f}")

                                   count    min  median    max
reference          phase   arm                                
S4 cost 10 Nov     offline S4          3  1.031   1.038  1.058
                   online  S4          3  1.032   1.047  1.061
feasibility 11 Nov offline M1          4  0.864   0.991  0.999
                           S0-cpp      6  1.005   1.013  1.019
                           S0-go       6  1.006   1.015  1.023
                           S1          4  1.003   1.014  1.020
                           S2          4  0.996   1.013  1.020
                           S3          2  1.000   1.005  1.010

all compared cells: ratio grid / earlier from 0.864 to 1.061, median 1.014


In [2]:
# S3 at 1 GiB offline: per-repetition server_preproc_s in every attempt, against 8 times the 128 MiB median
# (S3's preprocessing grows linearly with the database: the 1, 16 and 128 MiB medians are printed first).
run = raw / "all-20261112_tier1"
s3 = {db: st.median(json.loads(l)["metrics"]["server_preproc_s"] for l in (run / f"S3_{db}_1bit_offline.jsonl").read_text().splitlines()[1:])
      for db in ("1MiB", "16MiB", "128MiB")}
print("S3 offline medians, s:", {k: round(v, 1) for k, v in s3.items()})
expected = 8 * s3["128MiB"]
print(f"linear expectation at 1 GiB: {expected:.0f} s per repetition")
attempts = {"rejected: lid closed (1625 s asleep)": run / "rejected/S3_1GiB_1bit_offline.1794506562.jsonl",
            "rejected: rerun under load, timed out": run / "rejected/S3_1GiB_1bit_offline.timeout-under-load.jsonl",
            "final": run / "S3_1GiB_1bit_offline.jsonl"}
for name, f in attempts.items():
    if not f.exists():
        print(f"{name}: not run yet")
        continue
    rows = [json.loads(l) for l in f.read_text().splitlines()]
    reps = [(r["rep"], r["metrics"]["server_preproc_s"], r["metrics"]["server_preproc_cpu_s"]) for r in rows if r["status"] == "ok"]
    print(f"{name}: statuses {sorted({r['status'] for r in rows})}")
    for rep, s, cpu in reps:
        print(f"   rep {rep}: {s:7.0f} s wall, {cpu:7.0f} s CPU, {s / expected:.2f} x the linear expectation")

S3 offline medians, s: {'1MiB': 1.0, '16MiB': 14.6, '128MiB': 111.3}
linear expectation at 1 GiB: 890 s per repetition
rejected: lid closed (1625 s asleep): statuses ['ok']
   rep 0:    1055 s wall,    1051 s CPU, 1.19 x the linear expectation
rejected: rerun under load, timed out: statuses ['ok', 'timeout']
   rep 0:    1566 s wall,    1548 s CPU, 1.76 x the linear expectation
   rep 1:    2688 s wall,    2652 s CPU, 3.02 x the linear expectation
   rep 2:    2563 s wall,    2531 s CPU, 2.88 x the linear expectation
final: statuses ['ok', 'timeout']
   rep 0:    1396 s wall,    1385 s CPU, 1.57 x the linear expectation
   rep 1:    2509 s wall,    2474 s CPU, 2.82 x the linear expectation
   rep 2:    2725 s wall,    2685 s CPU, 3.06 x the linear expectation


## Memory: offline peaks against online peaks

Every offline phase runs one preprocessing per repetition, and every online phase runs one before its queries. Cell 3 lists the runner's peak footprint (summed over the adapter's process tree) for both phases. Cell 4 checks whether that sum counts a forked child's shared pages twice. Cell 5 splits one S1 offline repetition into the VeriSimplePIR adapter's parent and the child it forks (`run_offline_isolated`).

In [3]:
# Peak footprint per (cell, phase) from the ledger's last ok or oom attempt, offline against online.
G = 2**30
peak = {}
for line in (run / "ledger.jsonl").read_text().splitlines():
    d = json.loads(line)
    if d["status"] in ("ok", "oom"):
        peak[d["cell_id"]] = (d["peak_footprint_bytes"] / G, d["status"])
rows = []
for cell_id, (off, st_off) in peak.items():
    if cell_id.endswith("_offline") and (on := peak.get(cell_id.replace("_offline", "_online"))):
        arm, db, rec, _ = cell_id.split("_")
        rows.append({"arm": arm, "db": db, "record": rec, "offline_GiB": round(off, 2), "offline": st_off,
                     "online_GiB": round(on[0], 2), "online": on[1], "off/on": round(off / on[0], 2)})
mem = pd.DataFrame(rows)
big = mem[mem.db.isin(["128MiB", "1GiB"]) & ~mem.arm.isin(["S4"])]
print(big.sort_values(["arm", "db", "record"]).to_string(index=False))
ok = big[(big.offline == "ok") & (big.online == "ok")]
print("\noff/on where both phases fit, by arm:", ok.groupby("arm")["off/on"].agg(["min", "max"]).to_dict("index"))

   arm     db record  offline_GiB offline  online_GiB online  off/on
    M0 128MiB   1KiB         0.00      ok        0.26     ok    0.01
    M0 128MiB   256B         0.00      ok        0.26     ok    0.01
    M0 128MiB    32B         0.28      ok        0.29     ok    0.98
    M0 128MiB   8KiB         0.00      ok        0.26     ok    0.01
    M0   1GiB   1KiB         1.53      ok        2.02     ok    0.76
    M0   1GiB   256B         1.75      ok        2.04     ok    0.86
    M0   1GiB    32B         1.45      ok        2.26     ok    0.64
    M0   1GiB   8KiB         1.63      ok        2.02     ok    0.81
    M1 128MiB   1KiB         1.15      ok        1.03     ok    1.12
    M1 128MiB   256B         3.04      ok        1.91     ok    1.59
    M1 128MiB    32B        14.01     oom       11.65     ok    1.20
    M1 128MiB   8KiB         0.99      ok        0.57     ok    1.72
    M1   1GiB   1KiB         9.90      ok        9.21     ok    1.08
    M1   1GiB   256B        14.00 

In [4]:
# Does the runner's sum of ri_phys_footprint count a forked child's copy-on-write pages twice?
# Allocate and write 2 GiB, fork a child that only sleeps, and read both footprints and the system's memory in use.
import os, subprocess, sys, time
sys.path.insert(0, str(Path.cwd().parent / "src"))
from cs4406_sem_project.harness.memcap import footprint_bytes

def used():
    out = subprocess.run(["vm_stat"], capture_output=True, text=True).stdout
    page = int(out.split("page size of ")[1].split()[0])
    get = lambda k: int(next(l for l in out.splitlines() if l.startswith(k)).split(":")[1].strip(" ."))
    return page * sum(get(k) for k in ("Pages active", "Pages inactive", "Pages wired down",
                                        "Pages speculative", "Pages occupied by compressor"))

before = used()
buf = bytearray(os.urandom(1 << 20)) * 2048
time.sleep(1)
allocated = used()
r, w = os.pipe()
pid = os.fork()
if pid == 0:
    os.close(r); os.write(w, b"x"); time.sleep(6); os._exit(0)
os.read(r, 1); time.sleep(2)
fp_parent, fp_child, forked = footprint_bytes(os.getpid()), footprint_bytes(pid), used()
os.waitpid(pid, 0)
del buf
print(f"parent footprint {fp_parent / G:.2f} GiB, child footprint {fp_child / G:.2f} GiB")
print(f"system memory in use: +{(allocated - before) / G:.2f} GiB for the 2 GiB, +{(forked - allocated) / G:.2f} GiB after the fork")

parent footprint 2.08 GiB, child footprint 0.01 GiB
system memory in use: +1.99 GiB for the 2 GiB, +0.01 GiB after the fork


In [5]:
# One S1 offline repetition and one S1 online run at 128 MiB x 32 B (each with --reps 1 --warmup 0), sampling the
# parent's and its children's footprints every 0.1 s. Online runs its preprocessing in the parent; offline in a child.
import tempfile, psutil
repo = Path.cwd().parent
split = {}
with tempfile.TemporaryDirectory() as tmp:
    for phase in ("offline", "online"):
        argv = [str(repo / "build/adapters/vsp-adapter"), "--arm", "S1", "--db-bytes", str(128 << 20), "--record-bytes", "32",
                "--phase", phase, "--reps", "1", "--warmup", "0", "--seed", "1234", "--fault", "none",
                "--recovery", "abort", "--out", f"{tmp}/{phase}.jsonl"]
        proc = subprocess.Popen(argv, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        pp, par, chi, tot, at_fork = psutil.Process(proc.pid), 0, 0, 0, None
        while proc.poll() is None:
            try:
                kids = pp.children(recursive=True)
            except psutil.Error:
                break
            a, b = footprint_bytes(proc.pid), sum(footprint_bytes(k.pid) for k in kids)
            if kids and at_fork is None:
                at_fork = a
            par, chi, tot = max(par, a), max(chi, b), max(tot, a + b)
            time.sleep(0.1)
        assert proc.returncode == 0, proc.returncode
        split[phase] = (par, chi, tot)
        print(f"{phase:7s}: parent peak {par / G:.2f} GiB (at the fork {(at_fork or 0) / G:.2f}), "
              f"children peak {chi / G:.2f} GiB, sum peak {tot / G:.2f} GiB")
pre_child = split["offline"][1]
pre_inproc = split["online"][0] - split["offline"][0]
print(f"preprocessing adds {pre_inproc / G:.2f} GiB in the parent (online) and {pre_child / G:.2f} GiB in the forked child (offline)")

offline: parent peak 1.13 GiB (at the fork 1.13), children peak 1.57 GiB, sum peak 2.70 GiB


online : parent peak 2.03 GiB (at the fork 0.00), children peak 0.00 GiB, sum peak 2.03 GiB
preprocessing adds 0.90 GiB in the parent (online) and 1.57 GiB in the forked child (offline)


In [6]:
# S3's footprint. First the ledger's peak for every S3 1 GiB attempt (a timed-out attempt records none). Then one S3
# offline run at 128 MiB with one warm-up and two repetitions, sampling the footprint every 0.1 s and splitting the
# samples at the adapter's "offline rep" log lines: the footprint when each repetition starts, and its peak.
import threading
for e in (json.loads(l) for l in (run / "ledger.jsonl").read_text().splitlines()):
    if e["cell_id"].startswith("S3_1GiB"):
        pk = e.get("peak_footprint_bytes")
        print(f'{e["cell_id"]} at {e["at"][:16]}: {e["status"]}, {e["wall_s"]:.0f} s, peak',
              f"{pk / G:.2f} GiB" if pk else "not recorded")
with tempfile.TemporaryDirectory() as tmp:
    argv = [str(repo / "build/adapters/apir-adapter"), "--arm", "S3", "--db-bytes", str(128 << 20), "--record-bytes", "0",
            "--phase", "offline", "--reps", "2", "--warmup", "1", "--seed", "1234", "--fault", "none", "--out", f"{tmp}/rows.jsonl"]
    t0, marks, samples = time.monotonic(), [], []
    proc = subprocess.Popen(argv, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
    def read_log():
        for line in proc.stderr:
            if "offline rep" in line:
                marks.append(time.monotonic() - t0)
    reader = threading.Thread(target=read_log, daemon=True)
    reader.start()
    while proc.poll() is None:
        samples.append((time.monotonic() - t0, footprint_bytes(proc.pid)))
        time.sleep(0.1)
    reader.join()
    assert proc.returncode == 0, proc.returncode
    rows = [json.loads(l) for l in open(f"{tmp}/rows.jsonl")]
bounds = marks + [samples[-1][0] + 1]
for k, (start, end) in enumerate(zip(bounds, bounds[1:])):
    seg = [f for t, f in samples if start <= t < end]
    print(f"S3 128 MiB offline rep {k}: starts at {seg[0] / G:.2f} GiB, peaks at {max(seg) / G:.2f} GiB,",
          f"server_preproc_s {rows[k]['metrics']['server_preproc_s']:.1f}")

S3_1GiB_1bit_offline at 2026-11-12T18:02: hygiene_failed, 1444 s, peak 10.54 GiB
S3_1GiB_1bit_offline at 2026-11-12T20:03: timeout, 7201 s, peak not recorded
S3_1GiB_1bit_online at 2026-11-12T20:23: oom, 1247 s, peak 14.04 GiB
S3_1GiB_1bit_offline at 2026-11-13T00:04: timeout, 7201 s, peak not recorded


S3 128 MiB offline rep 0: starts at 1.27 GiB, peaks at 1.40 GiB, server_preproc_s 113.2
S3 128 MiB offline rep 1: starts at 1.80 GiB, peaks at 1.80 GiB, server_preproc_s 113.0
S3 128 MiB offline rep 2: starts at 1.94 GiB, peaks at 1.94 GiB, server_preproc_s 113.0
